# Лабораторная работа № 4.2

## Вариант 7: Логистика (Наследование и полиморфизм)

### Описание

В данной работе реализована иерархия классов для системы логистики с использованием:
- Наследования (дочерние классы `GroundShipping` и `AirShipping`)
- Полиморфизма (метод `calculate_delivery_cost()`)
- Композиции (класс-контейнер `DeliveryOrder`)

### Особенности реализации

- Использована функция `super()` для инициализации родительских классов
- Реализован полиморфный метод с разной логикой для наземной и авиадоставки
- Создан класс-контейнер для управления заказами
- Применена договорная скидка для бизнес-клиентов

### Автор

Гришина Анастасия, ЦИБ-251


In [1]:
# ==============================================================================
# Лабораторная работа № 4.2. Вариант 7: Логистика
# Тема: Наследование, Полиморфизм и Инкапсуляция
# Автор: Гришина Анастасия, ЦИБ-251
# ==============================================================================

# ==============================================================================
# БАЗОВЫЕ КЛАССЫ (из ЛР 4.1)
# ==============================================================================

class Shipment:
    """Базовый класс 1 (БК1): описывает груз (отправление) в логистике."""

    def __init__(self, shipment_id: int, weight: float):
        """Конструктор с валидацией данных через try-except."""
        try:
            if not isinstance(shipment_id, int) or shipment_id <= 0:
                raise ValueError("ID отправления должен быть положительным целым числом.")

            if not isinstance(weight, (int, float)) or weight <= 0:
                raise ValueError("Вес груза должен быть положительным числом.")

            self.shipment_id = shipment_id
            self.weight = float(weight)

        except ValueError as e:
            print(f"Ошибка инициализации груза: {e}")
            raise
        except TypeError as e:
            print(f"Ошибка типа данных при создании груза: {e}")
            raise

    def calculate_delivery_cost(self) -> float:
        """Полиморфный метод: расчет стоимости доставки.
        В базовом классе возвращает NotImplementedError (абстрактное поведение).
        """
        raise NotImplementedError(
            "Метод calculate_delivery_cost() должен быть реализован в дочернем классе."
        )

    def display_info(self):
        """Выводит подробную информацию о грузе."""
        print(f"--- Груз №{self.shipment_id} ---")
        print(f"Вес отправления: {self.weight} кг")
        print("-" * 30)

    def __str__(self):
        """Строковое представление груза."""
        return f"Груз #{self.shipment_id} | Вес: {self.weight} кг"


class Recipient:
    """Базовый класс 2 (БК2): описывает получателя груза."""

    def __init__(self, name: str, address: str):
        """Конструктор с валидацией данных через try-except."""
        try:
            if not isinstance(name, str) or not name.strip():
                raise ValueError("Имя получателя не может быть пустым.")

            if not isinstance(address, str) or not address.strip():
                raise ValueError("Адрес доставки не может быть пустым.")

            self.name = name.strip()
            self.address = address.strip()

        except ValueError as e:
            print(f"Ошибка инициализации получателя: {e}")
            raise
        except TypeError as e:
            print(f"Ошибка типа данных при создании получателя: {e}")
            raise

    def display_info(self):
        """Выводит подробную информацию о получателе."""
        print(f"--- Получатель: {self.name} ---")
        print(f"Адрес доставки: {self.address}")
        print("-" * 30)

    def __str__(self):
        """Строковое представление получателя."""
        return f"Получатель: {self.name} | Адрес: {self.address}"

# ==============================================================================
# ДОЧЕРНИЕ КЛАССЫ 1 (ДК1): наследуют от Shipment
# ==============================================================================

class GroundShipping(Shipment):
    """ДК1: Наземная доставка (автотранспорт).
    Тариф: 50 руб/кг.
    """
    RATE_PER_KG = 50.0  # руб/кг для наземной доставки

    def __init__(self, shipment_id: int, weight: float, vehicle_type: str):
        """Инициализация с использованием super() для общих полей."""
        super().__init__(shipment_id, weight)  # ← вызов конструктора родителя

        if not isinstance(vehicle_type, str) or not vehicle_type.strip():
            raise ValueError("Тип транспорта не может быть пустым.")
        self.vehicle_type = vehicle_type.strip()

    def calculate_delivery_cost(self) -> float:
        """Полиморфная реализация: стоимость = вес × тариф наземной доставки."""
        return self.weight * self.RATE_PER_KG

    def display_info(self):
        """Переопределение метода с добавлением типа транспорта."""
        print(f"--- Груз №{self.shipment_id} (Наземная доставка) ---")
        print(f"Транспорт: {self.vehicle_type}")
        print(f"Вес: {self.weight} кг | Тариф: {self.RATE_PER_KG} руб/кг")
        print(f"Стоимость доставки: {self.calculate_delivery_cost():.2f} руб")
        print("-" * 40)

    def __str__(self):
        return (f" Наземная #{self.shipment_id} | {self.vehicle_type} | "
                f"{self.weight} кг | {self.calculate_delivery_cost():.2f} руб")


class AirShipping(Shipment):
    """ДК1: Авиадоставка (самолет).
    Тариф: 150 руб/кг (в 3 раза дороже наземной).
    """
    RATE_PER_KG = 150.0  # руб/кг для авиадоставки

    def __init__(self, shipment_id: int, weight: float, airline: str):
        """Инициализация с использованием super() для общих полей."""
        super().__init__(shipment_id, weight)  # ← вызов конструктора родителя

        if not isinstance(airline, str) or not airline.strip():
            raise ValueError("Название авиакомпании не может быть пустым.")
        self.airline = airline.strip()

    def calculate_delivery_cost(self) -> float:
        """Полиморфная реализация: стоимость = вес × тариф авиадоставки."""
        return self.weight * self.RATE_PER_KG

    def display_info(self):
        """Переопределение метода с добавлением авиакомпании."""
        print(f"--- Груз №{self.shipment_id} (Авиадоставка) ---")
        print(f"Авиакомпания: {self.airline}")
        print(f"Вес: {self.weight} кг | Тариф: {self.RATE_PER_KG} руб/кг")
        print(f"Стоимость доставки: {self.calculate_delivery_cost():.2f} руб")
        print("-" * 40)

    def __str__(self):
        return (f"  Авиа #{self.shipment_id} | {self.airline} | "
                f"{self.weight} кг | {self.calculate_delivery_cost():.2f} руб")


# ==============================================================================
# ДОЧЕРНИЙ КЛАСС 2 (ДК2): наследует от Recipient
# ==============================================================================

class BusinessRecipient(Recipient):
    """ДК2: Бизнес-получатель с договорной скидкой на доставку."""

    def __init__(self, name: str, address: str, discount: float, contract_number: str):
        """Инициализация с использованием super() для общих полей."""
        super().__init__(name, address)  # ← вызов конструктора родителя

        try:
            if not isinstance(discount, (int, float)) or not (0 <= discount <= 100):
                raise ValueError("Скидка должна быть числом от 0 до 100 процентов.")

            if not isinstance(contract_number, str) or not contract_number.strip():
                raise ValueError("Номер договора не может быть пустым.")

            self.discount = float(discount)
            self.contract_number = contract_number.strip()

        except ValueError as e:
            print(f"Ошибка инициализации бизнес-получателя: {e}")
            raise

    def get_discount_multiplier(self) -> float:
        """Возвращает множитель для расчета итоговой стоимости (1 - скидка/100)."""
        return 1.0 - (self.discount / 100.0)

    def display_info(self):
        """Переопределение метода с выводом информации о скидке."""
        print(f"--- Бизнес-получатель: {self.name} ---")
        print(f"Адрес: {self.address}")
        print(f"Договор: {self.contract_number}")
        print(f"Договорная скидка: {self.discount}%")
        print("-" * 40)

    def __str__(self):
        return (f" {self.name} | Договор: {self.contract_number} | "
                f"Скидка: {self.discount}%")


# ==============================================================================
# КЛАСС-КОНТЕЙНЕР (Композиция)
# ==============================================================================

class DeliveryOrder:
    """Класс-контейнер: заказ на доставку.
    Содержит получателя (БК2/ДК2) и список грузов (БК1/ДК1).
    """

    def __init__(self, recipient: Recipient, order_id: int):
        """Конструктор контейнера.
        Args:
            recipient: объект получателя (обычный или бизнес).
            order_id: уникальный номер заказа.
        """
        if not isinstance(recipient, Recipient):
            raise TypeError("Получатель должен быть экземпляром класса Recipient.")
        if not isinstance(order_id, int) or order_id <= 0:
            raise ValueError("ID заказа должен быть положительным целым числом.")

        self.recipient = recipient
        self.order_id = order_id
        self._items: list[Shipment] = []  # ← приватный список грузов (инкапсуляция)

    def add_item(self, item: Shipment) -> None:
        """Добавляет объект груза в заказ."""
        if not isinstance(item, Shipment):
            raise TypeError("Можно добавить только объект класса Shipment или его наследника.")
        self._items.append(item)
        print(f" В заказ №{self.order_id} добавлен: {item}")

    def remove_item(self, shipment_id: int) -> bool:
        """Удаляет груз из заказа по его ID."""
        for item in self._items:
            if item.shipment_id == shipment_id:
                self._items.remove(item)
                return True
        return False

    def calculate_total(self) -> float:
        """Итоговый расчет стоимости доставки.
        1. Перебирает список грузов.
        2. Вызывает у каждого полиморфный метод calculate_delivery_cost().
        3. Если получатель — BusinessRecipient, применяет договорную скидку.
        """
        subtotal = 0.0
        for item in self._items:
            subtotal += item.calculate_delivery_cost()  # ← ПОЛИМОРФИЗМ

        # Применяем скидку, если получатель — бизнес-клиент
        if isinstance(self.recipient, BusinessRecipient):
            final_total = subtotal * self.recipient.get_discount_multiplier()
        else:
            final_total = subtotal

        return final_total

    def print_report(self) -> None:
        """Выводит детализированный отчет по заказу."""
        print("=" * 60)
        print(f" ЗАКАЗ НА ДОСТАВКУ №{self.order_id}")
        print("=" * 60)

        print("\n ИНФОРМАЦИЯ О ПОЛУЧАТЕЛЕ:")
        self.recipient.display_info()

        print(f"\n СОСТАВ ЗАКАЗА ({len(self._items)} позиций):")
        print("-" * 60)

        subtotal = 0.0
        for i, item in enumerate(self._items, start=1):
            item.display_info()
            cost = item.calculate_delivery_cost()
            subtotal += cost
            print(f"  → Позиция {i}: {cost:.2f} руб\n")

        print("-" * 60)
        print(f" Промежуточный итог: {subtotal:.2f} руб")

        if isinstance(self.recipient, BusinessRecipient):
            discount_amount = subtotal - self.calculate_total()
            print(f" Скидка по договору ({self.recipient.discount}%): -{discount_amount:.2f} руб")

        print(f" ИТОГО К ОПЛАТЕ: {self.calculate_total():.2f} руб")
        print("=" * 60)


# ==============================================================================
# ДЕМОНСТРАЦИЯ СЦЕНАРИЯ (Этап 5)
# ==============================================================================

if __name__ == "__main__":
    print(" ЗАПУСК ЛАБОРАТОРНОЙ РАБОТЫ № 4.2 (Вариант 7: Логистика)\n")

    try:
        # ======================================================================
        # Шаг 1. Создаем получателей (обычного и бизнес-клиента)
        # ======================================================================
        print(" ШАГ 1. Создание получателей...")
        regular_client = Recipient(
            name="Иван Иванов",
            address="г. Москва, ул. Ленина, д. 1"
        )

        business_client = BusinessRecipient(
            name="ООО 'Ромашка'",
            address="г. Санкт-Петербург, Невский пр., д. 10",
            discount=15.0,
            contract_number="LOG-2026-0042"
        )
        print(" Получатели созданы успешно.\n")

        # ======================================================================
        # Шаг 2. Создаем грузы разных типов (дочерние классы)
        # ======================================================================
        print(" ШАГ 2. Создание грузов (наземная и авиадоставка)...")
        cargo_ground_1 = GroundShipping(
            shipment_id=101,
            weight=25.5,
            vehicle_type="Фура Mercedes Actros"
        )
        cargo_ground_2 = GroundShipping(
            shipment_id=102,
            weight=150.0,
            vehicle_type="Газель Next"
        )
        cargo_air_1 = AirShipping(
            shipment_id=201,
            weight=4.2,
            airline="Аэрофлот"
        )
        cargo_air_2 = AirShipping(
            shipment_id=202,
            weight=12.8,
            airline="S7 Airlines"
        )
        print(" Грузы созданы успешно.\n")

        # ======================================================================
        # Шаг 3. Создаем заказы (контейнеры) и добавляем туда грузы
        # ======================================================================
        print(" ШАГ 3. Формирование заказов...")

        # Заказ №1: обычный клиент
        order_1 = DeliveryOrder(recipient=regular_client, order_id=1001)
        order_1.add_item(cargo_ground_1)
        order_1.add_item(cargo_air_1)

        # Заказ №2: бизнес-клиент (со скидкой)
        order_2 = DeliveryOrder(recipient=business_client, order_id=1002)
        order_2.add_item(cargo_ground_2)
        order_2.add_item(cargo_air_2)
        order_2.add_item(cargo_ground_1)
        print()

        # ======================================================================
        # Шаг 4. Выводим детализированные отчеты
        # ======================================================================
        print(" ШАГ 4. Детализированные отчеты по заказам:\n")
        order_1.print_report()
        print("\n")
        order_2.print_report()

        # ======================================================================
        # Шаг 5. Демонстрация полиморфизма в действии
        # ======================================================================
        print("\n ШАГ 5. Демонстрация полиморфизма (один интерфейс — разное поведение):")
        all_shipments = [cargo_ground_1, cargo_ground_2, cargo_air_1, cargo_air_2]
        for shipment in all_shipments:
            print(f"  {shipment}")

        print("\n Программа выполнена успешно!")

    except Exception as e:
        print(f"\n Критическая ошибка: {e}")
        print("Выполнение программы остановлено.")

 ЗАПУСК ЛАБОРАТОРНОЙ РАБОТЫ № 4.2 (Вариант 7: Логистика)

 ШАГ 1. Создание получателей...
 Получатели созданы успешно.

 ШАГ 2. Создание грузов (наземная и авиадоставка)...
 Грузы созданы успешно.

 ШАГ 3. Формирование заказов...
 В заказ №1001 добавлен:  Наземная #101 | Фура Mercedes Actros | 25.5 кг | 1275.00 руб
 В заказ №1001 добавлен:   Авиа #201 | Аэрофлот | 4.2 кг | 630.00 руб
 В заказ №1002 добавлен:  Наземная #102 | Газель Next | 150.0 кг | 7500.00 руб
 В заказ №1002 добавлен:   Авиа #202 | S7 Airlines | 12.8 кг | 1920.00 руб
 В заказ №1002 добавлен:  Наземная #101 | Фура Mercedes Actros | 25.5 кг | 1275.00 руб

 ШАГ 4. Детализированные отчеты по заказам:

 ЗАКАЗ НА ДОСТАВКУ №1001

 ИНФОРМАЦИЯ О ПОЛУЧАТЕЛЕ:
--- Получатель: Иван Иванов ---
Адрес доставки: г. Москва, ул. Ленина, д. 1
------------------------------

 СОСТАВ ЗАКАЗА (2 позиций):
------------------------------------------------------------
--- Груз №101 (Наземная доставка) ---
Транспорт: Фура Mercedes Actros
Вес: 25